In [1]:
#import library
from pandas_gbq import read_gbq
import pandas as pd
import numpy as np
import os
import datetime
import ssl
import logging

In [2]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Get today's date
today = date.today()

# --- Current month period (CY): last completed calendar month ---
# last day of previous month
end_date = date(today.year, today.month, 1) - timedelta(days=1)
# first day of that month
start_date = date(end_date.year, end_date.month, 1)

# --- Prior period (PP): previous calendar month (1 month) ---
pp_end_date = start_date - timedelta(days=1)
pp_start_date = date(pp_end_date.year, pp_end_date.month, 1)

# --- Corresponding periods last year (LY): same calendar months ---
# "CWLY" = same month last year (range)
cwly_start_date = start_date - relativedelta(years=1)
cwly_end_date   = end_date   - relativedelta(years=1)

# "PWLY" = prior-month period last year (range)
pwly_start_date = pp_start_date - relativedelta(years=1)
pwly_end_date   = pp_end_date   - relativedelta(years=1)

# --- YTD helpers ---
ytd_last_year = end_date - relativedelta(years=1)  # same day last year as this period end
begin_of_current_year = date(end_date.year, 1, 1)
begin_of_last_year    = date(end_date.year - 1, 1, 1)

# Output all dates (I return ranges for LY to match month logic)
(
    end_date, start_date,
    pp_end_date, pp_start_date,
    cwly_start_date, cwly_end_date,
    pwly_start_date, pwly_end_date,
    begin_of_last_year, ytd_last_year, begin_of_current_year
)


(datetime.date(2026, 3, 31),
 datetime.date(2026, 3, 1),
 datetime.date(2026, 2, 28),
 datetime.date(2026, 2, 1),
 datetime.date(2025, 3, 1),
 datetime.date(2025, 3, 31),
 datetime.date(2025, 2, 1),
 datetime.date(2025, 2, 28),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 3, 31),
 datetime.date(2026, 1, 1))

In [3]:
from custom_query import read_sql_query, sql_files

# Read SQL queries from files
queries = {key: read_sql_query(path) for key, path in sql_files.items()}

# Execute queries if all were successfully reada
if all(queries.values()):
    try:
        monthly_note = read_gbq(queries["weekly_note"], project_id='pcln-pl-airanalytics-prod')
        finance_data = read_gbq(queries["finance_data"], project_id='pcln-pl-airanalytics-prod')
        gds_incentives = read_gbq(queries["gds_incentives"], project_id='pcln-pl-airanalytics-prod')
        tsa_data = read_gbq(queries["tsa_data"], project_id='pcln-pl-airanalytics-prod')
        dau_conversion_data = read_gbq(queries["dau_conversion_query"], project_id='pcln-pl-airanalytics-prod')
        print("SQL queries executed successfully.")
        
    except Exception as e:
        print(f"Failed to execute SQL queries: {e}")

/Users/sye/Library/Python/3.9/lib/python/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=qUtfkeUUijJtCfbs8V17dZID9D3jMY&prompt=consent&access_type=offline
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
SQL queries executed successfully.


In [4]:
# make a copy of every data set
df_monthly=monthly_note.copy()
df_finance=finance_data.copy()
df_gds_incentive=gds_incentives.copy()
df_tsa=tsa_data.copy()
df_monthly.columns = df_monthly.columns.str.lower()
dau_conversion=dau_conversion_data.copy()

## Summary Table

In [5]:

def format_number(num):
    if pd.isna(num):
        return ''
    if abs(num) >= 1e6:
        return f"{num/1e6:.1f}M"
    elif abs(num) >= 1e3:
        return f"{num/1e3:.0f}K"
    elif abs(num) < 1e3:
        return "<1K"
    return f"{num:.0f}"


def format_percentage(x, decimals=1, multiply_100=True):
    def fmt(v):
        if pd.isna(v):
            return ''
        v = float(v)
        if multiply_100:
            v *= 1
        return f"{v:.{decimals}f}%"

    if isinstance(x, pd.Series):
        return x.map(fmt)
    if isinstance(x, pd.DataFrame):
        return x.applymap(fmt) 
    return fmt(x)  


def format_percentage_2(num):
    if pd.isna(num):
        return ''
    return f"{num:.2f}%"

def round_to_nearest_10(num):
  return round(num / 10) * 10

df_pricelince=df_monthly[(df_monthly['brand']== 'Priceline')]
df_pricelince_air=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['offer_type']== 'Flights Only')]
df_pricelince_b2c=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['company']== 'Priceline B2C')]
df_pricelince_b2c_standalone=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['offer_type']== 'Flights Only')&(df_monthly['company']== 'Priceline B2C')]


In [6]:
df_pricelince_b2c_standalone

,trans_date,us_travel_type,company,brand,offer_type,carrier_detail,carrier,offer_method_code,search_channel,search_channel_group,...,net_contr_fee_cy,yoy_net_contr_fee,net_orders_cy,net_orders_ly,gr_wex_fee_cy,gr_wex_fee_ly,ref_wex_fee_cy,ref_wex_fee_ly,net_wex_fee_cy,net_wex_fee_ly
349,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,ANA All Nippon Airways (NH),Other,Retail (Disclosed),SEM Brand,Web Marketing,...,9.96,-9.96,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Commission Junction,Affiliate,...,0.00,0.00,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
351,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Email,Direct,...,0.00,0.00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
352,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Meta,Shop PPC,...,0.00,0.00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
353,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Meta,Shop PPC,...,0.00,3.10,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923151,2026-04-01,X,Priceline B2C,Priceline,Flights Only,interCaribbean (JY),Other,Retail (Disclosed),Other,Shop PPC,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
923152,2026-04-01,X,Priceline B2C,Priceline,Flights Only,interCaribbean (JY),Other,Retail (Disclosed),SEM Brand,Web Marketing,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
923153,2026-04-01,X,Priceline B2C,Priceline,Flights Only,interCaribbean (JY),Other,Retail (Disclosed),SEM Brand,Web Marketing,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
923154,2026-04-01,X,Priceline B2C,Priceline,Flights Only,interCaribbean (JY),Other,Retail (Disclosed),SEM Core,Web Marketing,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
import kpi_month
import importlib

importlib.reload(kpi_month)

from kpi_month import calculate_business_metrics

# Calculate business metrics
df_business = calculate_business_metrics(
    df_pricelince,
    start_date,
    end_date, 
    pp_start_date,
    pp_end_date, 
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date, 
    format_number)


from kpi_month import calculate_carrier_metrics
# Calculate carrier metrics
df_carrier = calculate_carrier_metrics(
    df_pricelince_b2c_standalone,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date,   
    format_number,
)


from kpi_month import calculate_channel_metrics
df_channel = calculate_channel_metrics(
    df_pricelince_b2c_standalone,
    start_date, end_date,
    pp_start_date, pp_end_date,
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date, 
    format_number,
)

In [8]:
df_business

,Net Tickets_standalone,YoY_standalone,YoY PW_standalone,Net Tickets_package,YoY_package,YoY PW_package,Net Tickets_total,YoY_total,YoY PW_total
B2C,712K,3.2%,10.7%,161K,12.0%,41.6%,872K,4.7%,15.5%
B2B,31K,-18.0%,-10.8%,27K,-18.6%,5.2%,58K,-18.3%,-3.2%
Total,743K,2.1%,9.6%,188K,6.3%,34.1%,930K,2.9%,14.1%


In [9]:
df_channel

,Net Tickets_App,YoY_App,YoY PW_App,Net Tickets_Desk/MWEB,YoY_Desk/MWEB,YoY PW_Desk/MWEB,Net Tickets_Total,YoY_Total,YoY PW_Total
Direct,148K,-2.0%,11.3%,96K,-20.4%,-18.4%,244K,-10.2%,-2.4%
Web Marketing,49K,129.8%,146.5%,269K,17.0%,21.0%,318K,26.6%,30.3%
Shop PPC,26K,-12.9%,4.7%,76K,-9.9%,4.8%,103K,-10.7%,4.7%
Affiliate,1K,98.1%,242.2%,45K,-11.2%,-4.0%,47K,-9.6%,-1.8%
Total,224K,10.5%,23.7%,487K,0.1%,5.6%,712K,3.2%,10.7%


In [10]:
df_carrier

,Net Tickets_Retail,YoY_Retail,YoY PW_Retail,Net Tickets_Opaque,YoY_Opaque,YoY PW_Opaque,Net Tickets_Total,YoY_Total,YoY PW_Total
American Airlines (AA),126K,-15.1%,-7.8%,12K,-62.84568163860694%,-76.11951674987614%,138K,-23.8%,-20.5%
Delta Air Lines (DL),91K,-16.4%,-1.1%,1K,-60.68941504178274%,-75.28898022090932%,92K,-17.6%,-4.4%
United Airlines (UA),72K,-6.2%,4.0%,39K,17.702005555725144%,20.789944828090178%,111K,0.9%,9.0%
Southwest Airlines (WN),99K,28860.2%,20477.7%,1K,143500.0%,<NA>%,100K,29194.5%,20687.0%
Spirit Airlines (NK),29K,-55.7%,-52.0%,,,,29K,-55.7%,-52.0%
Frontier Airlines (F9),97K,47.2%,60.5%,<1K,<NA>%,<NA>%,97K,47.2%,60.5%
Alaska Airlines (AS),24K,19.9%,41.5%,16K,17.838976892893466%,52.71207371520852%,41K,19.1%,46.0%
JetBlue Airways (B6),29K,-12.2%,16.9%,<1K,11000.0%,<NA>%,29K,-11.8%,17.3%
Other,74K,-16.2%,-11.4%,<1K,-20.654396728016355%,41.55844155844155%,75K,-16.3%,-11.2%
Total,642K,5.7%,14.3%,70K,-15.370213793936461%,-15.712358100378399%,712K,3.2%,10.7%


##  DAU


In [11]:
import importlib
import dau_roi_table_month

importlib.reload(dau_roi_table_month)

from dau_roi_table_month import calculate_dau_conversion

df_dau_conversion = calculate_dau_conversion(
    dau_conversion,
    format_percentage,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    cwly_start_date,
    cwly_end_date,
    pwly_start_date,
    pwly_end_date,
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year)

df_dau_conversion

,DAU,DAU_pw,DAU_cwly,DAU_pwly,DAU_ytd,DAU_ytd_ly,DAU YoY,DAU YoY_PW,DAU YoY_YTD
channel,,,,,,,,,
Affiliate,66315,58223,32511,26982,178084,89318,104.0%,115.8%,99.4%
Direct,3999832,3454184,3843747,3364606,11039289,11231791,4.1%,2.7%,-1.7%
SEM Brand,714121,632799,665062,586152,1941604,1955413,7.4%,8.0%,-0.7%
SEM Core,2582259,2148567,1626097,1397508,6831706,4990664,58.8%,53.7%,36.9%
Shop PPC Cheapflights,1129159,1075479,962690,900246,3112565,3000552,17.3%,19.5%,3.7%
Shop PPC Google,3129,2607,2100,1596,7950,5940,49.0%,63.3%,33.8%
Shop PPC Kayak,1057275,757701,169193,152771,2449064,536775,524.9%,396.0%,356.3%
Shop PPC Others,702035,543875,595777,532263,1784451,1705431,17.8%,2.2%,4.6%
Total,10254125,8673435,7897177,6962124,27344713,23515884,29.8%,24.6%,16.3%


# Summary Table

## YTD

In [12]:

import importlib
import summary_table_actual_month  

importlib.reload(summary_table_actual_month)

from summary_table_actual_month import create_finance_number_month


finance_number = create_finance_number_month(
    df_monthly,
    df_gds_incentive,
    start_date, end_date,                 
    pp_start_date, pp_end_date,           
    cwly_start_date, cwly_end_date,       
    pwly_start_date, pwly_end_date,       
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    format_percentage,
    format_number,
)

finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,930073.0,824831.0,903939.0,723198.0,2.891124,14.053274,2586390.0,2449462.0,5.590126
1,Gross Tickets,1004131.0,889270.0,969872.0,781837.0,3.532322,13.741099,2787622.0,2637923.0,5.674881
2,Net Revenue (net_contribution),8885880.0,7805434.0,9126913.0,7552579.0,-2.640904,3.347926,25043060.0,26398674.0,-5.135158
3,Gross Revenue (gross_contribution),9732835.0,8582911.0,10106326.0,8438193.0,-3.695618,1.715029,27393119.0,29293725.0,-6.488101
4,Normalized Net Tickets,803146.0,709422.0,771808.0,621612.0,4.060336,14.126175,2229947.0,2103523.0,6.010108
5,Normalized Gross Tickets,864602.0,762928.0,827476.0,671941.0,4.486656,13.540921,2397131.0,2265604.0,5.805383
6,Net Cont + Fee,10121450.0,8870087.0,9949657.0,8260482.0,1.726616,7.379779,28466419.0,28725921.0,-0.903373
7,Gross Cont + Fee,11060312.0,9716480.0,10992480.0,9203698.0,0.617079,5.571478,31050617.0,31806276.0,-2.375818
8,GDS Incentive,456068.0,449588.0,919380.0,799232.0,-50.393994,-43.747522,1388613.0,2585183.0,-46.285698
9,VCC Rebate (net_wex_fee),562403.0,428476.0,238727.0,112517.0,135.584439,280.809539,1380378.0,482810.0,185.905237


In [13]:
#current week finance data
#net tickets by scenario
plan_net_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)

#previous week finance data
#net tickets by scenario
plan_net_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


#ytd week finance data
#net tickets by scenario
plan_net_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)



In [14]:
period_order = ["CW", "PW", "YTD", "YTD_LY"]

# revenue as Series (no squeeze)
plan_grrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= end_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= start_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

# 1) MultiIndex columns (metric, period)
plan_metrics_mi = pd.concat(
    {
        ("net_tkts", "CW"): plan_net_cw,
        ("net_tkts", "PW"): plan_net_pw,
        ("net_tkts", "YTD"): plan_net_ytd,
        # ("net_tkts", "YTD_LY"): plan_net_ytd_ly,

        ("gr_tkts", "CW"): plan_gr_cw,
        ("gr_tkts", "PW"): plan_gr_pw,
        ("gr_tkts", "YTD"): plan_gr_ytd,
        # ("gr_tkts", "YTD_LY"): plan_gr_ytd_ly,

        ("gr_rev", "CW"): plan_grrev_cw,
        ("gr_rev", "PW"): plan_grrev_pw,
        ("gr_rev", "YTD"): plan_grrev_ytd,
        # ("gr_rev", "YTD_LY"): plan_grrev_ytd_ly,

        ("net_rev", "CW"): plan_netrev_cw,
        ("net_rev", "PW"): plan_netrev_pw,
        ("net_rev", "YTD"): plan_netrev_ytd,
        # ("net_rev", "YTD_LY"): plan_netrev_ytd_ly,
    },
    axis=1
)
plan_metrics_mi.index.name = "scenario"

# 2) reshape to "period columns"
plan_metrics_period_cols = (
    plan_metrics_mi
      .stack(0)                # stack metric level -> rows
      .reset_index()
      .rename(columns={"level_1": "metric"})
)

# optional formatting
if callable(format_number):
    for c in period_order:
        if c in plan_metrics_period_cols.columns:
            plan_metrics_period_cols[c] = plan_metrics_period_cols[c].round(0)

keep = ["scenario", "metric"] + [c for c in period_order if c in plan_metrics_period_cols.columns]
plan_metrics_period = plan_metrics_period_cols[keep]

# filter PLAN
plan_metrics_period_plan = plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN")]
plan_metrics_period_plan

/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_6305/3762684958.py:65: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  plan_metrics_mi


,scenario,metric,CW,PW,YTD
0,PLAN,gr_rev,9448061.0,7856731.0,25774024.0
1,PLAN,gr_tkts,982514.0,800843.0,2666414.0
2,PLAN,net_rev,8892750.0,7385935.0,24261058.0
3,PLAN,net_tkts,921745.0,751189.0,2503770.0


In [15]:
# import importlib
# import build_plan_metrics
# importlib.reload(build_plan_metrics)
# from build_plan_metrics import build_plan_metrics_period_plan



# plan_metrics_period_plan =  build_plan_metrics_period_plan(
#     df_finance,
#     start_date,
#     end_date,
#     pp_start_date,
#     begin_of_current_year,
#     begin_of_last_year,
#     ytd_last_year,
#     format_number=None,
#     period_order=None,
#     scenario_value="PLAN",
# )

In [16]:

import importlib
import summary_table_actual_month  

importlib.reload(summary_table_actual_month)

from summary_table_actual_month import create_finance_number_month


finance_number = create_finance_number_month(
    df_monthly,
    df_gds_incentive,
    start_date, end_date,                 
    pp_start_date, pp_end_date,           
    cwly_start_date, cwly_end_date,       
    pwly_start_date, pwly_end_date,       
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    format_percentage,
    format_number,
)

finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,930073.0,824831.0,903939.0,723198.0,2.891124,14.053274,2586390.0,2449462.0,5.590126
1,Gross Tickets,1004131.0,889270.0,969872.0,781837.0,3.532322,13.741099,2787622.0,2637923.0,5.674881
2,Net Revenue (net_contribution),8885880.0,7805434.0,9126913.0,7552579.0,-2.640904,3.347926,25043060.0,26398674.0,-5.135158
3,Gross Revenue (gross_contribution),9732835.0,8582911.0,10106326.0,8438193.0,-3.695618,1.715029,27393119.0,29293725.0,-6.488101
4,Normalized Net Tickets,803146.0,709422.0,771808.0,621612.0,4.060336,14.126175,2229947.0,2103523.0,6.010108
5,Normalized Gross Tickets,864602.0,762928.0,827476.0,671941.0,4.486656,13.540921,2397131.0,2265604.0,5.805383
6,Net Cont + Fee,10121450.0,8870087.0,9949657.0,8260482.0,1.726616,7.379779,28466419.0,28725921.0,-0.903373
7,Gross Cont + Fee,11060312.0,9716480.0,10992480.0,9203698.0,0.617079,5.571478,31050617.0,31806276.0,-2.375818
8,GDS Incentive,456068.0,449588.0,919380.0,799232.0,-50.393994,-43.747522,1388613.0,2585183.0,-46.285698
9,VCC Rebate (net_wex_fee),562403.0,428476.0,238727.0,112517.0,135.584439,280.809539,1380378.0,482810.0,185.905237


In [17]:
def build_vs_plan(finance_number: pd.DataFrame, plan_metrics_period: pd.DataFrame,format_percentage) -> pd.DataFrame:
    plan = (plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN"),
                                    ["metric", "CW", "PW", "YTD"]]
            .set_index("metric")
            .rename(columns={"CW": "CW_plan", "PW": "PW_plan", "YTD": "YTD_plan"}))

    actual = (finance_number[["Measure", "CW", "PW", "CY"]]
              .set_index("Measure"))

    measure_to_metric = {
        "Net Tickets": "net_tkts",
        "Gross Tickets": "gr_tkts",
        "Net Cont + Fee + Incentives + vcc rebate(Flight Only)": "net_rev",
        # "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)": "gr_rev",
    }

    tmp = actual.rename(index=measure_to_metric)
    tmp = tmp.join(plan, how="right")
    
    metric_order = list(measure_to_metric.values())
    tmp = tmp.reindex(metric_order)
    def vs_pct(a, p):
        p = p.replace(0, np.nan)
        return (a / p - 1) * 100

    out = pd.DataFrame(index=tmp.index)
    out["Reporting Week (vs Plan)"] = vs_pct(tmp["CW"], tmp["CW_plan"])
    out["Previous Week (vs Plan)"]  = vs_pct(tmp["PW"], tmp["PW_plan"])
    out["YTD (vs Plan)"]            = vs_pct(tmp["CY"], tmp["YTD_plan"])  # CY = YTD actual

    metric_to_measure = {v: k for k, v in measure_to_metric.items()}
    out.index = out.index.map(metric_to_measure)

    for c in ["Reporting Week (vs Plan)", "Previous Week (vs Plan)", "YTD (vs Plan)"]:
        out[c] = out[c].apply(format_percentage)

    return out



In [18]:
finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,930073.0,824831.0,903939.0,723198.0,2.891124,14.053274,2586390.0,2449462.0,5.590126
1,Gross Tickets,1004131.0,889270.0,969872.0,781837.0,3.532322,13.741099,2787622.0,2637923.0,5.674881
2,Net Revenue (net_contribution),8885880.0,7805434.0,9126913.0,7552579.0,-2.640904,3.347926,25043060.0,26398674.0,-5.135158
3,Gross Revenue (gross_contribution),9732835.0,8582911.0,10106326.0,8438193.0,-3.695618,1.715029,27393119.0,29293725.0,-6.488101
4,Normalized Net Tickets,803146.0,709422.0,771808.0,621612.0,4.060336,14.126175,2229947.0,2103523.0,6.010108
5,Normalized Gross Tickets,864602.0,762928.0,827476.0,671941.0,4.486656,13.540921,2397131.0,2265604.0,5.805383
6,Net Cont + Fee,10121450.0,8870087.0,9949657.0,8260482.0,1.726616,7.379779,28466419.0,28725921.0,-0.903373
7,Gross Cont + Fee,11060312.0,9716480.0,10992480.0,9203698.0,0.617079,5.571478,31050617.0,31806276.0,-2.375818
8,GDS Incentive,456068.0,449588.0,919380.0,799232.0,-50.393994,-43.747522,1388613.0,2585183.0,-46.285698
9,VCC Rebate (net_wex_fee),562403.0,428476.0,238727.0,112517.0,135.584439,280.809539,1380378.0,482810.0,185.905237


In [19]:
plan_metrics_period

,scenario,metric,CW,PW,YTD
0,PLAN,gr_rev,9448061.0,7856731.0,25774024.0
1,PLAN,gr_tkts,982514.0,800843.0,2666414.0
2,PLAN,net_rev,8892750.0,7385935.0,24261058.0
3,PLAN,net_tkts,921745.0,751189.0,2503770.0


In [20]:
df_vs_plan = build_vs_plan(finance_number, plan_metrics_period, format_percentage)
df_vs_plan

,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
metric,,,
Net Tickets,0.9%,9.8%,3.3%
Gross Tickets,2.2%,11.0%,4.5%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),5.0%,7.2%,5.9%


In [21]:
finance_number
wanted = [
    "Net Tickets",
    "Gross Tickets",
    "Net Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Normalized Net Tickets",
    "Normalized Gross Tickets",
]

finance_output = finance_number.loc[
    finance_number["Measure"].isin(wanted),
    ["Measure", "CW", "Reporting Week", "Previous Week", "YTD"]
].reset_index(drop=True)

# ---- formatting ----
num_cols = ["CW"]
pct_cols = ["Reporting Week", "Previous Week", "YTD"]

if callable(format_number):
    for c in num_cols:
        finance_output[c] = finance_output[c].apply(format_number)

if callable(format_percentage):
    for c in pct_cols:
        finance_output[c] = finance_output[c].apply(format_percentage)

finance_output=finance_output.set_index('Measure')
finance_output=finance_output.rename(columns={"CW": "Actual"})
finance_output=finance_output.reindex(wanted).fillna('')
finance_output


,Actual,Reporting Week,Previous Week,YTD
Measure,,,,
Net Tickets,930K,2.9%,14.1%,5.6%
Gross Tickets,1.0M,3.5%,13.7%,5.7%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),9.3M,0.6%,3.9%,-2.9%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),10.2M,0.3%,2.8%,-3.8%
Normalized Net Tickets,803K,4.1%,14.1%,6.0%
Normalized Gross Tickets,865K,4.5%,13.5%,5.8%


## TSA

In [22]:
tsa_cy=df_tsa[(df_tsa['date']>=start_date)&(df_tsa['date']<=end_date)]['tsa_passengers'].sum()
pcln_cy=df_tsa[(df_tsa['date']>=start_date)&(df_tsa['date']<=end_date)]['pcln_passengers'].sum()
tsa_actual_cy=(pcln_cy*100/tsa_cy)

tsa_pw=df_tsa[(df_tsa['date']>=pp_start_date)&(df_tsa['date']<=pp_end_date)]['tsa_passengers'].sum()
pcln_pw=df_tsa[(df_tsa['date']>=pp_start_date)&(df_tsa['date']<=pp_end_date)]['pcln_passengers'].sum()
tsa_actual_pw=(pcln_pw*100/tsa_pw)

tsa_cwly=df_tsa[(df_tsa['date']>=cwly_start_date)&(df_tsa['date']<=cwly_end_date)]['tsa_passengers'].sum()
pcln_cwly=df_tsa[(df_tsa['date']>=cwly_start_date)&(df_tsa['date']<=cwly_end_date)]['pcln_passengers'].sum()
tsa_actual_cwly=(pcln_cwly*100/tsa_cwly)


tsa_pwly=df_tsa[(df_tsa['date']>=pwly_start_date)&(df_tsa['date']<=pwly_end_date)]['tsa_passengers'].sum()
pcln_pwly=df_tsa[(df_tsa['date']>=pwly_start_date)&(df_tsa['date']<=pwly_end_date)]['pcln_passengers'].sum()
tsa_actual_pwly=(pcln_pwly*100/tsa_pwly)


tsa_ytd=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_current_year)]['tsa_passengers'].sum()
pcln_ytd=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_current_year)]['pcln_passengers'].sum()
tsa_actual_ytd=(pcln_ytd*100/tsa_ytd)


tsa_ytd_ly=df_tsa[(df_tsa['date'] <=ytd_last_year)&(df_tsa['date']>=begin_of_last_year)]['tsa_passengers'].sum()
pcln_ytd_ly=df_tsa[(df_tsa['date']<=ytd_last_year)& (df_tsa['date']>=begin_of_last_year)]['pcln_passengers'].sum()
tsa_actual_ytd_ly=(pcln_ytd_ly*100/tsa_ytd_ly)

print("TSA CY",tsa_cy,"PCLN CY",pcln_cy,"%TSA market share",tsa_actual_cy)
print("TSA LY",tsa_cwly,"PCLN LY",pcln_cwly,"%TSAmarket share",tsa_actual_cwly)
print("TSA PW",tsa_pw,"PCLN PW",pcln_pw,"%TSAmarket share",tsa_actual_pw)
print("TSA PWLY",tsa_pwly,"PCLN PWly",pcln_pwly,"%TSAmarket share",tsa_actual_pwly)
print("TSA YTD CY",tsa_ytd,"PCLN YTD CY",pcln_ytd,"%TSAmarket share CY YTD ",tsa_actual_ytd)
print("TSA YTD LY",tsa_ytd_ly,"PCLN YTD LY",pcln_cwly,"%TSA market share LY YTD ",tsa_actual_ytd_ly)

TSA CY 78543126 PCLN CY 1104642.0 %TSA market share 1.406414610999822
TSA LY 77229641 PCLN LY 974730.0 %TSAmarket share 1.2621190353584577
TSA PW 64256259 PCLN PW 869328.0 %TSAmarket share 1.3529078933773595
TSA PWLY 62590333 PCLN PWly 801335.0 %TSAmarket share 1.28028556742141
TSA YTD CY 208660296 PCLN YTD CY 2898477.0 %TSAmarket share CY YTD  1.3890888949951457
TSA YTD LY 205597207 PCLN YTD LY 974730.0 %TSA market share LY YTD  1.3096350088063209


In [23]:
# df_summary=df_summary.reset_index(names=['Metric'])

df_summary= pd.DataFrame()
df_summary.loc['DAU','Actual']=format_number(df_dau_conversion.loc[:, 'DAU'].loc['Total'])
df_summary.loc['DAU','Reporting Week']=df_dau_conversion.loc[:, 'DAU YoY'].loc['Total']
df_summary.loc['DAU','Previous Week']=df_dau_conversion.loc[:, 'DAU YoY_PW'].loc['Total']
df_summary.loc['DAU','YTD']=df_dau_conversion.loc[:, 'DAU YoY_YTD'].loc['Total']


df_summary.loc['TSA','Actual']=format_percentage_2(tsa_actual_cy)
df_summary.loc['TSA','Reporting Week']=format_percentage(((tsa_actual_cy/tsa_actual_cwly)-1)*100)
df_summary.loc['TSA','Previous Week']=format_percentage((tsa_actual_pw/tsa_actual_pwly-1)*100)
df_summary.loc['TSA','YTD']=format_percentage((tsa_actual_ytd/tsa_actual_ytd_ly-1)*100)

df_summary=df_summary.fillna('')

df_summary

,Actual,Reporting Week,Previous Week,YTD
DAU,10.3M,29.8%,24.6%,16.3%
TSA,1.41%,11.4%,5.7%,6.1%


In [24]:
final_summary = pd.concat([finance_output,df_summary])
final_summary = final_summary.join(df_vs_plan, how="left").fillna('')
final_summary

,Actual,Reporting Week,Previous Week,YTD,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
Net Tickets,930K,2.9%,14.1%,5.6%,0.9%,9.8%,3.3%
Gross Tickets,1.0M,3.5%,13.7%,5.7%,2.2%,11.0%,4.5%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),9.3M,0.6%,3.9%,-2.9%,5.0%,7.2%,5.9%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),10.2M,0.3%,2.8%,-3.8%,,,
Normalized Net Tickets,803K,4.1%,14.1%,6.0%,,,
Normalized Gross Tickets,865K,4.5%,13.5%,5.8%,,,
DAU,10.3M,29.8%,24.6%,16.3%,,,
TSA,1.41%,11.4%,5.7%,6.1%,,,


In [25]:

import config_table
import importlib

importlib.reload(config_table)
from docx import Document
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Pt

from config_table import (
    clear_document,
    set_font,
    create_word_table,
    create_summary_table,
    create_others_table,
    # create_dau_table,
    # create_roi_table
)


# Data preparation and configurations
carrier_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Retail.jpg', 'title': 'Retail'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Express.jpg', 'title': 'Express Deals'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

business_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Standalone.jpg', 'title': 'Standalone'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Package.jpg', 'title': 'Package'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

channel_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/App.jpg', 'title': 'App'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/MWeb Desktop.jpg', 'title': 'Web'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

source_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Published.jpg', 'title': 'Published'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Private.jpg', 'title': 'Private'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

def customize_bullet_style(paragraph, font_size):
    """
    Customize bullet style in a paragraph using XML manipulation.
    """
    # Access the paragraph's properties
    pPr = paragraph._element.get_or_add_pPr()
    numPr = OxmlElement('w:numPr')
    ilvl = OxmlElement('w:ilvl')
    ilvl.set(qn('w:val'), '0')  # Set indentation level
    numId = OxmlElement('w:numId')
    numId.set(qn('w:val'), '1')  # Set numbering ID

    numPr.append(ilvl)
    numPr.append(numId)
    pPr.append(numPr)

    # Modify font properties
    for run in paragraph.runs:
        run.font.name = "Montserrat"
        run.font.size = font_size

# Create a Word Document
word_document = Document()

# Clear the document (if necessary)
clear_document(word_document)

# Add Title
word_document.add_paragraph()
word_document.add_paragraph(f'Summary')
set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
word_document.add_paragraph('\n')

# Create Summary Table
create_summary_table(word_document,final_summary)

# Add General Notes
summary_notes = [
    'All Priceline tickets (includes B2B, Package, Express Deals, Phone Sales)',
    'Normalized tickets are counted the same as tickets, except split tickets count as 1 instead of 2',
    'Refunds are assigned to refund date',
    'Revenue is contribution w/fee + GDS incentives + VCC rebates.  Does not include package or phone sales.',
    'Daily Active Users: engaged customers in GA4',
    'TSA footprint: travel date; numerator counts all slices (OW is 1 slice; RT is 2 slices) where the first segment is either US domestic or US outbound (Priceline-only); denominator includes all people passing through TSA screening machines'
]
for note in summary_notes:
    word_document.add_paragraph(note, style='List Bullet')
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)
    customize_bullet_style(word_document.paragraphs[-1], font_size=Pt(7))

# Add Tables for Business, Carrier, Channel, and Source
tables = [
    (df_business, "Business", business_logos, [
        'All Priceline tickets (includes Express Deals and Phone Sales, not normalized)'
    ]),
    (df_carrier, "Carrier", carrier_logos, [
        'Priceline (including phone sales), B2C, Standalone (not normalized)'
    ]),
    (df_channel, "Channel", channel_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)'
    ])
]
for df, title, logos, notes in tables:
    if df is None:
        continue

   # Add specific custom notes for each table
    if title == "Business":
        custom_note = "Total Business - Detail"
    elif title == "Carrier":
        custom_note = "Priceline B2C Standalone - Detail"
    else:
        custom_note = " "

    # Add the custom note before the table
    word_document.add_paragraph(custom_note)
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
    word_document.add_paragraph()

    create_word_table(df, title, logos, word_document)

    # word_document.add_paragraph()

    for note in notes:
        word_document.add_paragraph(note, style='List Bullet')
        # word_document.add_paragraph('\n')
        set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)


# Save the Document
output_filename = os.path.join('../montly_output/', f'Flight Performance Month Ending {start_date}.docx')
word_document.save(output_filename)
print(f"Word document '{output_filename}' has been created successfully.")

# Save PDF to Shared Drive
share_drive_path = '../../../Flight Weekly Note Output/'
word_document.save(share_drive_path + f'Flight Performance Month Ending {start_date}.pdf')

print(f"Word document saved to shared drive at '{share_drive_path}' has been created successfully.")


Word document '../montly_output/Flight Performance Month Ending 2026-03-01.docx' has been created successfully.
Word document saved to shared drive at '../../../Flight Weekly Note Output/' has been created successfully.
